In [ ]:
#Import necessary libraries (e.g., pandas, numpy)
import pandas as pd
import nltk
import torch
import torch.nn as nn
import math

from collections import Counter
from torch.utils.data import random_split
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ceb1003\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
# Part 1
# load data
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
# test
train_df.head()

,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [3]:
# inspect data
print(train_df.columns)
print(train_df['Class Index'].value_counts())

Index(['Class Index', 'Title', 'Description'], dtype='str')
Class Index
3    30000
4    30000
2    30000
1    30000
Name: count, dtype: int64


In [4]:
# combine title and description text
train_df['text'] = train_df['Title'] + " " + train_df['Description']
test_df['text'] = test_df['Title'] + " " + test_df['Description']

In [5]:
# Part 2: Text preprocessing
# lowercase text & tokenize text
def tokenize(text):
    return nltk.word_tokenize(text.lower())
train_tokens = train_df['text'].apply(tokenize)

In [ ]:
# build a vocabulary from the training set
vocab = Counter()
for tokens in train_tokens:
    vocab.update(tokens)
# Keep top words
vocab = {
    word: i + 2
    for i, (word, _) in enumerate(vocab.most_common(20000))
}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

In [7]:
# convert tokens to integer IDs
def encode(tokens):
   return [vocab.get(word, 1) for word in tokens]

In [ ]:
# pad sequences to a fixed maximum length
MAX_LEN = 128

def pad(seq):
   if len(seq) < MAX_LEN:
       return seq + [0]*(MAX_LEN - len(seq))
   return seq[:MAX_LEN]

def collate_fn(batch):
    texts, labels = zip(*batch)

    texts = torch.stack(texts)
    labels = torch.tensor(labels)

    return texts, labels

In [ ]:
# Part 3: Dataset and DataLoader
# a custom Dataset and a collate_fn for padding
class NewsDataset(Dataset):

    def __init__(self, df):
        self.texts = df['text'].apply(tokenize).apply(encode).apply(pad)
        self.labels = df['Class Index'] - 1

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        return (
            torch.tensor(self.texts.iloc[idx], dtype=torch.long),
            torch.tensor(self.labels.iloc[idx], dtype=torch.long)
        )
    
# Create full training dataset
full_train_dataset = NewsDataset(train_df)

# Split into train and validation
train_size = int(0.9 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size]
)

# Test dataset
test_dataset = NewsDataset(test_df)

# DATALOADERS GO HERE

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    collate_fn=collate_fn
)

In [11]:
# Part 4: Implement the Transformer classifier
# PositionalEncoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=128):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0)/d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [12]:
# TransformerClassifier
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, 128)
        self.pos_encoding = PositionalEncoding(128)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=4,
            dim_feedforward=256,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc = nn.Linear(128, 4)
        
    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        
        x = self.transformer(x)
        
        x = x.mean(dim=1)  # pooling
        return self.fc(x)

In [ ]:
# Part 5: Train the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TransformerClassifier(len(vocab)).to(device)
# cross-entropy loss
criterion = nn.CrossEntropyLoss()
# Adam optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    model.train()
    total_loss = 0

    # mini-batch gradient descent
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")
    # Validation
    model.eval()

    val_preds = []
    val_true = []

    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)

            outputs = model(X)
            preds = outputs.argmax(dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_true.extend(y.cpu().numpy())

    val_acc = accuracy_score(val_true, val_preds)

    print(f"Validation Accuracy: {val_acc:.4f}")

Epoch 1, Loss: 1620.8158


In [ ]:
# Part 6: Evaluate the model
model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)

        outputs = model(X)
        preds = outputs.argmax(dim=1).cpu().numpy()

        y_pred.extend(preds)
        y_true.extend(y.numpy())

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, average='macro'))
print("Recall:", recall_score(y_true, y_pred, average='macro'))
print("F1:", f1_score(y_true, y_pred, average='macro'))

print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))